In [1]:
!pip install sentence-transformers datasets torch transformers accelerate

Defaulting to user installation because normal site-packages is not writeable


In [2]:
from datasets import load_dataset

def load_nq_german(data_file = "./data/ng_german.jsonl.gz"):
    # Load the JSONL file as a dataset
    dataset = (
        load_dataset("json", data_files=data_file, split="train",num_proc=8)
        .remove_columns(["query", "answer"])
        .rename_column("question_de", "query")
        .rename_column("answer_de", "answer")
    )
    dataset_dict = dataset.train_test_split(test_size=1_000, seed=12)
    return dataset_dict


/home/openai/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import logging
import random

import numpy
import torch
#from torch import mps  # noqa: F401
#torch.mps.device = mps
from datasets import Dataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss, CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

In [4]:
logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)
random.seed(12)
torch.manual_seed(12)
numpy.random.seed(12)

In [5]:
# Feel free to adjust these variables:
use_prompts = True
include_prompts_in_pooling = True

# 1. Load a model to finetune with 2. (Optional) model card data
base_model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

In [6]:
model = SentenceTransformer(
    base_model_name,
    #tokenizer_kwargs={"max_seq_length": 512},
    model_card_data=SentenceTransformerModelCardData(
        language="de",
        license="apache-2.0",
        model_name=f"{base_model_name} trained on german Natural Questions pairs",
    ),
).to(torch.bfloat16)

2025-09-08 19:07:57 - Use pytorch device_name: cuda:0
2025-09-08 19:07:57 - Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


In [7]:
model.set_pooling_include_prompt(include_prompts_in_pooling)

In [8]:
# 2. (Optional) Define prompts
if use_prompts:
    query_prompt = "query: "
    corpus_prompt = "document: "
    prompts = {
        "query": query_prompt,
        "answer": corpus_prompt,
    }

In [ ]:
# 3. Load a dataset to finetune on
dataset_dict = load_nq_german()
train_dataset: Dataset = dataset_dict["train"]
eval_dataset: Dataset = dataset_dict["test"]


Setting num_proc from 8 back to 1 for the train split to disable multiprocessing as it only contains one shard.
2025-09-08 19:08:14 - Setting num_proc from 8 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Generating train split: 100188 examples [00:01, 77676.29 examples/s]


In [10]:
# 4. Define a loss function
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=16) 
#loss = MultipleNegativesRankingLoss(model) # <- this does work with mps (Apple Silicon)


In [ ]:
# 5. (Optional) Specify training arguments

# todo: limit train size for testing

run_name = "nq-german-" + base_model_name.split("/")[-1]
if use_prompts:
    run_name += "-prompts"
if not include_prompts_in_pooling:
    run_name += "-exclude-pooling-prompts"
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir=f"models/{run_name}",
    # Optional training parameters:
    num_train_epochs=0.25, # limit training to 1/4 epoch for development
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    learning_rate=4e-5,
    warmup_ratio=0.1,   
    fp16=False,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=True,  # Set to True if you have a GPU that supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=0.5,
    save_strategy="steps",
    save_steps=0.5,
    save_total_limit=2,
    logging_steps=5,
    logging_first_step=True,
    run_name=run_name,  # Will be used in W&B if `wandb` is installed
    seed=12,
    prompts=prompts if use_prompts else None,
)

In [16]:
# 7. Create a trainer & train
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    #evaluator=dev_evaluator,
)
trainer.train()

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# 8. Save the trained model
model.save_pretrained(f"models/{run_name}/final")

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer(f"./models/{run_name}/final")

# The sentences to encode
sentences = [
    "query: Das ist eine Frage über Kekse.",
    "answer: Das hier ist ein Keksrezept",
    "query: Ich mag Möven, oder?",
    "answer: Ich bin Paul die Möve",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)


similarities = model.similarity(embeddings, embeddings)
print(similarities)

In [ ]:
import numpy as np
# convert files for tensorflow embedding projector

# Save as TSV
np.savetxt('output.tsv', embeddings, delimiter='\t', fmt='%g')

with open("description.tsv","w") as f:
    f.writelines(sentences)